# GPBSF PointNet++ (SSG) 双卡极速训练 Notebook (Kaggle T4 x 2) - 纯 Python 实时流式版

- **全链路纯 Python 架构**：彻底淘汰 `%%bash` 块缓冲，解压、自检、双卡训练与评估日志均以行缓冲实时流式输出。
- **双 GPU 并发流式监控**：GPU 0 训练 Seed 3407，GPU 1 训练 Seed 3408，两张卡的每一轮 Epoch 进度实时交替打印。
- **全量 RAM 预载 + JIT 算子**：零小文件 I/O 延迟，C++ 执行图闭环加速。
- **全自动断点续跑**：每轮持久化 `last.pt`，意外中断后重跑自动无缝接力。

In [ ]:
#@title 1. 纯 Python 工作区初始化、代码解压与数据集自动链接
import os
import shutil
import zipfile
from pathlib import Path

WORKSPACE = Path("/kaggle/working/gpbsf")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORKSPACE))
print(f"=== 工作区已就绪: {WORKSPACE} ===", flush=True)

# 1. 自动查找并解压代码
print("正在扫描代码资源...", flush=True)
code_zips = list(Path("/kaggle/input").rglob("*code*.zip"))
code_markers = list(Path("/kaggle/input").rglob("run_classification_matrix.py"))

if code_zips:
    target_code_zip = code_zips[0]
    print(f"找到代码压缩包: {target_code_zip}，正在解压...", flush=True)
    with zipfile.ZipFile(target_code_zip, 'r') as zf:
        zf.extractall(WORKSPACE)
elif code_markers:
    project_root = code_markers[0].resolve().parents[1]
    print(f"找到代码源目录: {project_root}，正在复制...", flush=True)
    shutil.copytree(str(project_root), str(WORKSPACE), dirs_exist_ok=True)
else:
    raise FileNotFoundError("在 /kaggle/input 未找到代码压缩包或源码目录，请检查 Input 数据集挂载。")

# 2. 自动查找并挂载/解压数据集
data_target = WORKSPACE / "data" / "bgspcd_v4_robust"
data_target.mkdir(parents=True, exist_ok=True)
print("正在扫描数据集资源...", flush=True)
manifest_markers = list(Path("/kaggle/input").rglob("manifest.jsonl"))
dataset_zips = list(Path("/kaggle/input").rglob("*dataset*.zip")) + list(Path("/kaggle/input").rglob("*bgspcd*.zip"))

if manifest_markers:
    source_data_dir = manifest_markers[0].parent
    print(f"找到已挂载的数据集目录: {source_data_dir}，正在创建软链接...", flush=True)
    for item in source_data_dir.iterdir():
        target_link = data_target / item.name
        if target_link.exists() or target_link.is_symlink():
            target_link.unlink()
        target_link.symlink_to(item)
elif dataset_zips:
    target_dataset_zip = dataset_zips[0]
    print(f"找到数据集压缩包: {target_dataset_zip}，正在解压...", flush=True)
    with zipfile.ZipFile(target_dataset_zip, 'r') as zf:
        zf.extractall(data_target)
else:
    raise FileNotFoundError("在 /kaggle/input 未找到 manifest.jsonl 或数据集压缩包，请检查数据集挂载。")

manifest_check = data_target / "manifest.jsonl"
if not manifest_check.exists():
    # 兼容解压在子目录的情况
    nested = list(data_target.rglob("manifest.jsonl"))
    if nested:
        nested_dir = nested[0].parent
        for f in nested_dir.iterdir():
            shutil.move(str(f), str(data_target / f.name))

if not manifest_check.exists():
    raise FileNotFoundError(f"未能正确就绪 manifest.jsonl 于 {data_target}")

print(f"✅ 代码与数据集初始化完成！manifest 文件路径: {manifest_check} (大小: {manifest_check.stat().st_size / 1024:.1f} KB)", flush=True)

In [ ]:
#@title 2. 双 GPU 环境与 PointNet++ JIT 模型快速自检
import torch
import os
from pathlib import Path

os.chdir("/kaggle/working/gpbsf")
print(f"PyTorch 版本: {torch.__version__} | CUDA 可用: {torch.cuda.is_available()}", flush=True)
assert torch.cuda.is_available(), "未检测到 CUDA，请确认右上角 Accelerator 已选择 GPU T4 x 2"

device_count = torch.cuda.device_count()
print(f"可用 GPU 数量: {device_count}", flush=True)
for i in range(device_count):
    print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}", flush=True)

from experiments.classification.adapters.pointnet2 import PointNet2Adapter
adapter = PointNet2Adapter(Path('.'), torch.device('cuda:0'))
dummy = torch.randn(2, 2048, 3, device='cuda:0')
out = adapter.logits(dummy)
print(f"✅ PointNet++ JIT 向量化模型前向测试通过，输出张量形状: {out.shape}", flush=True)

In [ ]:
#@title 3. 双 GPU 纯 Python 实时流式并发训练、评估与打包 (Seed 3407 & 3408)
import subprocess
import time
import os
from pathlib import Path

WORKSPACE = Path("/kaggle/working/gpbsf")
os.chdir(str(WORKSPACE))

# 确保配置文件中 num_workers 为 0 (全量 RAM 常驻内存切片，零 IPC 开销最快)
cfg_path = WORKSPACE / "config/experiments/classification_v4_robust.json"
content = cfg_path.read_text(encoding="utf-8")
import json
cfg_data = json.loads(content)
cfg_data["training"]["num_workers"] = 0
cfg_path.write_text(json.dumps(cfg_data, indent=2), encoding="utf-8")
print("✅ 配置确认: training.num_workers = 0 (纯内存切片)", flush=True)

# 启动后台双卡并发训练
cmd_3407 = "CUDA_VISIBLE_DEVICES=0 python3 -m experiments.classification.cli --config config/experiments/classification_v4_robust.json train --model pointnet2 --seed 3407"
cmd_3408 = "CUDA_VISIBLE_DEVICES=1 python3 -m experiments.classification.cli --config config/experiments/classification_v4_robust.json train --model pointnet2 --seed 3408"

print("=== 启动双 GPU 并发极速训练 (Seed 3407 & 3408) ===", flush=True)
p1 = subprocess.Popen(cmd_3407, shell=True)
p2 = subprocess.Popen(cmd_3408, shell=True)
print(f"进程状态: GPU 0 (PID {p1.pid}) | GPU 1 (PID {p2.pid})", flush=True)

log_3407 = WORKSPACE / "runs/classification_v4_robust/pointnet2/seed_3407/training.log"
log_3408 = WORKSPACE / "runs/classification_v4_robust/pointnet2/seed_3408/training.log"

# 纯 Python 行流式监视并强制 flush 输出到前端界面
seen_3407, seen_3408 = 0, 0
while p1.poll() is None or p2.poll() is None:
    time.sleep(3)
    if log_3407.exists():
        lines_7 = log_3407.read_text(encoding="utf-8", errors="ignore").splitlines()
        while seen_3407 < len(lines_7):
            print(f"[GPU 0] {lines_7[seen_3407]}", flush=True)
            seen_3407 += 1
    if log_3408.exists():
        lines_8 = log_3408.read_text(encoding="utf-8", errors="ignore").splitlines()
        while seen_3408 < len(lines_8):
            print(f"[GPU 1] {lines_8[seen_3408]}", flush=True)
            seen_3408 += 1

p1.wait()
p2.wait()
print("🎉 双卡 125 轮模型训练全部圆满完成！", flush=True)

# 并行执行测试集评估
print("=== 正在执行双卡并行测试集评估 ===", flush=True)
eval_p1 = subprocess.Popen("CUDA_VISIBLE_DEVICES=0 python3 -m experiments.classification.cli --config config/experiments/classification_v4_robust.json evaluate --run-dir runs/classification_v4_robust/pointnet2/seed_3407 --split test", shell=True)
eval_p2 = subprocess.Popen("CUDA_VISIBLE_DEVICES=1 python3 -m experiments.classification.cli --config config/experiments/classification_v4_robust.json evaluate --run-dir runs/classification_v4_robust/pointnet2/seed_3408 --split test", shell=True)
eval_p1.wait()
eval_p2.wait()

# 汇总两组种子的指标
print("=== 正在汇总两组种子指标 ===", flush=True)
subprocess.run("python3 -m experiments.classification.cli --config config/experiments/classification_v4_robust.json aggregate --runs-dir runs/classification_v4_robust", shell=True, check=True)

# 打包最终成果
output_archive = "/kaggle/working/pointnet2_results_v4_robust.tar.gz"
subprocess.run(f"tar -czvf {output_archive} -C /kaggle/working/gpbsf runs/classification_v4_robust", shell=True, check=True)
print(f"🚀 成果压缩包已成功生成: {output_archive} (大小: {Path(output_archive).stat().st_size / (1024*1024):.2f} MB)", flush=True)